# conv-padding-zero — ex1: build a zero-padded 1-D input by slice assignment

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `conv-padding-zero`. Running the final beacon cell reports progress against the `CNN: Conv zero padding` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: Conv zero padding` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`conv-padding-zero`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "conv-padding-zero"
DD_SUBTOPIC = "CNN: Conv zero padding"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Zero-padding a conv input — quick refresher

Convolution shrinks spatial dimensions. To preserve them (or to give the kernel something to multiply against at the boundary), we **pad** the input with zeros before convolving.

**1-D form.** Given `x: (B, IC, W)` and pad amounts `left, right`:
```
x_padded.shape == (B, IC, left + W + right)
x_padded[:, :, left : left + W] == x
x_padded[:, :, :left]            == 0
x_padded[:, :, left + W:]        == 0
```

**Implementation idioms.**
- Manual: `x_padded = x.new_full((B, IC, left + W + right), 0.0); x_padded[:, :, left:left+W] = x`.
- Functional: `F.pad(x, (left, right))` — last-axis-first ordering, careful.

**Why zero specifically.** Zero is the **additive identity** for the kernel's dot product — padded cells contribute nothing to the output. For *max-pool*, by contrast, padding must be `-inf` (the additive identity of max), not zero, or padded cells will spuriously win the max.

### Exercise 1 — build a zero-padded 1-D input by slice assignment

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply the manual zero-padding pattern: allocate a zero-filled destination tensor sized to fit the padded input, then copy the original into the interior via slice assignment.
> Keywords: padding, slice-assign, boundary
> ```

**KCs targeted:** `pad-allocate-zero-buffer`, `pad-slice-assign-interior`

Implement `ex1_pad1d_zeros(x, left, right)`.

- `x` has shape `(B, IC, W)`.
- `left`, `right` are non-negative ints.
- Return shape `(B, IC, left + W + right)`. Entries before column `left` and after column `left + W` must be exactly zero; entries in the interior must equal the corresponding columns of `x`.

**Required approach.** Build the output by hand (so the mechanics are visible), not via `F.pad`:

1. Allocate a zero buffer with `x.new_zeros(B, IC, left + W + right)` (this respects `x`'s dtype and device).
2. Use slice assignment to copy `x` into columns `[left : left + W]` of the buffer.

Edge cases the test exercises:
- `left == 0` and `right == 0` (no-op, output should equal input).
- `left > 0`, `right == 0` (front-only pad).
- `left == 0`, `right > 0` (back-only pad).
- Non-trivial values in `x` (no accidental zeroing of interior).

In [ ]:
def ex1_pad1d_zeros(x: Tensor, left: int, right: int) -> Tensor:
    """Return x padded with `left` and `right` zeros along the last axis."""
    raise NotImplementedError()


def _test_ex1():
    # Basic: pad (1, 2, 3) → length 5+1+2 = 8.
    x = t.tensor([[[1.0, 2.0, 3.0, 4.0, 5.0]]])  # (1, 1, 5)
    out = ex1_pad1d_zeros(x, 1, 2)
    expected = t.tensor([[[0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 0.0, 0.0]]])
    assert out.shape == (1, 1, 8), f'expected (1,1,8), got {tuple(out.shape)}'
    assert out.dtype == x.dtype
    assert t.allclose(out, expected, atol=1e-6), f'values wrong:\n{out}\nvs\n{expected}'

    # Boundary cells must be exactly zero (not just close to zero).
    assert (out[..., :1] == 0).all(), 'left boundary must be exact 0'
    assert (out[..., -2:] == 0).all(), 'right boundary must be exact 0'

    # No-op: left=right=0.
    noop = ex1_pad1d_zeros(x, 0, 0)
    assert noop.shape == x.shape
    assert t.allclose(noop, x), 'pad(0, 0) must equal x'

    # Front-only.
    front = ex1_pad1d_zeros(x, 3, 0)
    assert front.shape == (1, 1, 8)
    assert (front[..., :3] == 0).all(), 'front 3 must be 0'
    assert t.allclose(front[..., 3:], x), 'remainder must match x'

    # Back-only.
    back = ex1_pad1d_zeros(x, 0, 4)
    assert back.shape == (1, 1, 9)
    assert (back[..., 5:] == 0).all(), 'back 4 must be 0'
    assert t.allclose(back[..., :5], x), 'front must match x'

    # Multi-channel, batched: must not flatten across B / IC.
    rng = t.Generator().manual_seed(0)
    x2 = t.randn(2, 3, 4, generator=rng)
    out2 = ex1_pad1d_zeros(x2, 2, 1)
    assert out2.shape == (2, 3, 7)
    # Interior matches.
    assert t.allclose(out2[:, :, 2:6], x2, atol=1e-6), 'interior must equal x2'
    # Boundaries zero in every (b, c) slice.
    assert (out2[:, :, :2] == 0).all(), 'left zeros must hold across batch+channels'
    assert (out2[:, :, 6:] == 0).all(), 'right zeros must hold across batch+channels'

    # Cross-check against F.pad on a random tensor (must agree exactly).
    from torch.nn import functional as F
    x3 = t.randn(1, 2, 7, generator=rng)
    left, right = 3, 5
    ours = ex1_pad1d_zeros(x3, left, right)
    ref  = F.pad(x3, (left, right), value=0.0)
    assert t.allclose(ours, ref), 'must agree with F.pad(value=0.0)'

    # Conv-readiness: passing the padded tensor through F.conv1d with no
    # extra padding must equal F.conv1d(x, weight, padding=(left, right))
    # when left == right.
    k = 3
    weight = t.randn(2, 2, k, generator=rng)
    pad = 2
    padded = ex1_pad1d_zeros(x3, pad, pad)
    y_manual = F.conv1d(padded, weight)
    y_native = F.conv1d(x3, weight, padding=pad)
    assert t.allclose(y_manual, y_native, atol=1e-5), 'pre-padded conv should equal native padded conv'
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_pad1d_zeros(x: Tensor, left: int, right: int) -> Tensor:
    B, IC, W = x.shape
    out = x.new_zeros(B, IC, left + W + right)
    out[..., left : left + W] = x
    return out
```

**Why `new_zeros` instead of `t.zeros`.** `x.new_zeros(...)` inherits `x`'s dtype and device automatically. `t.zeros(...)` defaults to `float32` on CPU regardless of `x`, which silently promotes/demotes when you assign into it.

**Slice assignment vs `F.pad`.** Both produce identical output. Slice assignment is more explicit (the slice indices `[left : left + W]` show *where* the original lives in the padded tensor). `F.pad(x, (left, right))` is more idiomatic in PyTorch code — its argument is in *reverse* axis order (last-axis pad comes first), which is a common bug source.

**Why this matters for conv.** ARENA's `conv1d`-from-scratch pre-pads the input rather than threading `padding` through the windowing math — the as_strided windowing formula becomes `ow = (padded_w - kw) // stride + 1` with no padding term. This is the canonical 'pad first, then window' pattern.

**Pad value for maxpool is not zero.** For `max_pool` you'd want `x.new_full((B, IC, total_w), -t.inf)` — zero would let padded cells win the max for negative-valued inputs.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()